# RAG 평가 개요
- RAG 평가란 RAG 시스템이 주어진 입력에 대해 얼마나 효과적으로 관련 정보를 검색하고, 이를 기반으로 정확하고 유의미한 응답을 생성하는지를 측정하는 과정이다. 
- **평가 요소**
    - **검색 단계 평가**
        - 입력 질문에 대해 검색된 문서나 정보의 관련성과 정확성을 평가.
    - **생성 단계 평가**
        - 검색된 정보를 기반으로 생성된 응답의 품질, 정확성등을 평가.
- **평가 방법**
    - **온/오프라인 평가**
        1. **오프라인 평가**
            - 미리 준비된 데이터셋을 활용하여 RAG 시스템의 성능을 측정한다.
        2. **온라인 평가**
            - 실제 사용자 트래픽과 피드백을 기반으로 시스템의 실시간 성능을 평가한다.
    - **정량적/정성적 평가**
        1. 정량적 평가
            - 자동화된 지표를 사용하여 생성된 텍스트의 품질을 평가한다.
        2. 정성적 평가
            - 전문가나 일반 사용자가 직접 생성된 응답의 품질을 평가하여 주관적인 지표를 평가한다.

# [RAGAS](https://www.ragas.io/)
- RAGAS는 RAG 파이프라인을 **정량적으로 평가하는** 오픈소스 프레임 워크이다. 
- RAGAS 문서: https://docs.ragas.io/en/stable/
## 설치
- `pip install ragas rapidfuzz`

## RAGAS 평가 지표 개요
![ragas_score](figures/ragas_score.png)
- **Generation**
    - llm 모델이 생성한 답변에 대한 평가 지표들.
    - **Faithfulness(신뢰성)**
        -  생성된 답변과 검색된 문서(context)간의 관련성을 평가하는 지표
        -  생성된 답변이 주어진 문맥(context)에 얼마나 충실한지를 평가하는 지표로 할루시네이션에 대한 평가로 볼 수있다.
    - **Answer relevancy(답변 적합성)**
        - 생성된 답변과 사용자의 질문간의 관련성을 평가하는 지표
        - 생성된 답변이 사용자의 질문과 얼마나 관련성이 있는지를 평가하는 지표.
- **Retrieval**
    -  질문에 대해 검색한 문서(context)들에 대한 평가
    -  **Context Precision(문맥 정밀도)**
        -  검색된 문서(context)들 중 질문과 관련 있는 것들이 **얼마나 상위 순위에 위치하는지** 평가하는 지표.
    -  **Context Recall(문맥 재현률)**
        -  검색된 문서(context)가 정답(ground-truth)의 정보를 얼마나 포함하고 있는지 평가하는 지표.
- 이러한 지표들은 RAG 파이프라인의 성능을 다각도로 평가하는 데 활용된다.
![RAGAS_score2](figures/RAGAS_score2.png)

## 주요 평가지표
### Generation 평가
- LLM이 생성한 답변에 대한 평가
  
#### Faithfulness (신뢰성)
- 생성된 답변이 얼마나 주어진 검색 문서들(context)를 잘 반영해서 생성되었는지 평가한다. 할루시네이션에 대한 평가라고 할 수 있다. 
- 점수범위: **0 ~ 1** (1에 가까울수록 좋음)
- 답변에 포함된 모든 주장이 context에서 얼마나 추출 가능한지를 확인한다.

##### 평가 방법
1. Answer에서 주장 구문(claim statement)들을 생성(추출)한다. (주장이란, 질문(user input)과 관련된 내용)
    - 예) 
        - **질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요? 
        - **LLM 답변**: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 3000만명이다.
2. 각 주장들을 context로 부터 추론 가능한지 판단한다. 이를 바탕으로 faithfulness 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다. .... 한국의 인구는 5000만명이고 서울에 1000만이 살고 있다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 추론가능한 주장.
            - 한국의 인구는 3000만명이다. -> context에서 추론 불가능한 주장.
3. **Faithfulness score** 를 계산한다. 총 주장 수 중에서 context로 부터 추론가능한 주장의 개수.    
    - 예)
        - Faithfulness Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 유추할 수있다.)
    - LLM 답변에서 주장을 추출 하는 것과 각 주장이 context에서 추론 가능한 지를 판단하는 것은 LLM 을 활용한다.
- 공식
    $$
    \text{Faithfulness Score}\;=\;\cfrac{\text{주어진\;context\;에서\;추론할\;수\;있는\;주장의\;개수}}{\text{총\;주장\;개수}}
    $$

### Answer relevancy (답변 적합성)
- 생성된 답변이 질문(user input)에 얼마나 잘 부합하는 지를 평가한다.
- 점수 범위: -1~1 (1에 가까울수록 좋음)
- LLM이 생성한 답변을 기반으로 질문들을 생성한다. 이렇게 생성한 질문들과 실제 질문(user input) 간의 유사도를 측정한다.

#### 평가방법
1. LLM이 생성한 답변을 기반으로 질문들을 생성한다.
    - 예) 
        - **LLM** 답변: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
2. 실제 질문과 생성한 질문간의 코사인 유사도를 측정한다. 그 평균이 최종 점수가 된다.
    - 예)
        - **실제 질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요?
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
- 공식
  $$
    \cfrac{1}{N} \sum_{i=1}^{N} \text{cosine\_similarity}(q_{\text{user}_{_i}}, q_{\text{generated}})
  $$

## Retrieval 평가
Vector store에서 검색한 context에 대한 평가

### Context Precision
- 검색된 문서(context)들 중 질문과 관련 있는 것들이 얼마나 **상위 순위**에 있는 지 평가.
- 점수 범위: 0~1 (1에 가까울수록 좋음)


#### 평가방법

- 공식
$$
 \text{Context\;Precision@K} = \frac{\sum_{k=1}^{K} \left( \text{Precision@k} \times v_k \right)}{\ 상위\;K개\;결과에서의\;관련\;항목\;수}
$$
$$
 \text{Precision@k} = \frac{\text{True\;positive@k}}{(\text{True\;positive@k} + \text{False\;positive@k})} \\
$$
- $\text{Precision@k}$: 개별 문서에 대한 Precision
- K: context 의 개수(chuck 수)
- $v_k$: 관련성 여부로 0 또는 1. (0: 관련 없음, 1: 관련 있음)

#### 예시
- 질문과 context 관련성의 예
    - 질문: 한국의 수도는 어디이고 인구는 얼마나 되나요?
    - **높은 정밀도 context들**: 질문과 직접적인 관련이 있는 문서들
        - 한국의 수도는 서울이고 인구는 5000만명 입니다. 
        - 한국의 수도는 서울입니다.
        - 한국은 동아시아에 위치해 있는 국가로 수도는 서울입니다.
        - 한국의 인구는 5000만명 입니다.
    - **낮은 정밀도 context**: 한국과 관련있어 검색될 수 있지만 질문과 직접적 관련이 없다. 
        - 한국은 동아시아에 위치한 국가입니다.
        - 한국의 K-pop은 전 세계적으로 유명합니다.
        - 비빔밥, 불고기는 한국의 대표적인 음식입니다.
    - **높은 정밀도의 context이 상위 순위에 위치했으면 높은 점수를 받는다.**

- 점수 계산 예:
    - **상위 5개의 검색 결과 중 1번째, 3번째, 4번째 문서가 관련이 있다고 가정하자.**
    - **Precision@K 계산**
        ```bash
            Precision@1 = 1/1 = 1.0    # True positive@1/(True positive@1 + False positive@1).  1/1(1번 문서 계산 시에는 1개 문서만 있으므로 분모가 1이 된다.)
            Precision@2 = 1/2 = 0.5
            Precision@3 = 2/3 ≈ 0.67    
            Precision@4 = 3/4 = 0.75
            Precision@5 = 3/5 = 0.6
        ```
    - **vk의 값**
        - 1번째: $v_1 = 1$ - 관련있음
        - 2번째: $v_2 = 0$ - 관련없음
        - 3번째: $v_3 = 1$ - 관련있음
        - 4번째: $v_4 = 1$ - 관련있음
        - 5번째: $v_5 = 0$ - 관련없음

    - **Context Precision@5**
        $$
        \text{Context\;Precision@5} = \frac{(1.0 \times 1) + (0.5 \times 0) + (0.67 \times 1) + (0.75 \times 1) + (0.6 \times 0)}{3} = \frac{1.0 + 0 + 0.67 + 0.75 + 0}{3} ≈ 0.807
        $$

### Context Recall (문맥 재현률)
- 검색된 문서(context)가 얼마나 정답(ground-truth)의 정보를 포함있는 지 평가하는 지표
- 점수 범위: 0~1 (1에 가까울수록 좋음)
- **정답(ground truth)의 각 주장(claim)이 검색된 context와 얼마나 일치**하는지 계산함.

#### 평가방법
1. 정답에서 주장(claim)들을 생성(추출)한다.
    - 예) 
        - **정답**: 한국의 수도는 서울이고 인구수는 5000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 5000만명이다.
2. 각 주장(claim)의 정보를 검색된 contexts에서 찾을 수 있는지 판별한다. 이를 바탕으로 context recall 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 찾을 수 있다.
            - 한국의 인구는 5000만명이다. -> context에서 찾을 수 없다.
3. **Context Recall Score** 를 계산한다. 총 주장 수 중에서 context로 부터 찾을 수 있는 주장의 개수.
    - 예)
        - Context Recall Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 찾을 수 있다.)

- 공식
    $$
    \text{Context Recall Score}\;=\;\cfrac{\text{GT의\;주장\;중\;주어진\;context\;에서\;찾을\;수\;있는\;주장의\;개수}}{\text{GT의\;총\;주장\;개수}}
    $$ 

# RAGAS 평가 실습

In [ ]:
# !uv pip install ragas rapidfuzz
# 설치 후 커널 재시작

In [ ]:
# docker run -p 6333:6333 -p 6334:6334   -v qdrant_storage:/qdrant/storage   qdrant/qdrant

In [1]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, SparseVectorParams, VectorParams
from langchain_openai import OpenAIEmbeddings

from dotenv import load_dotenv

load_dotenv()


True

In [2]:
# ##############################################################
# 데이터 준비
##############################################################

def load_and_split_olympic_data(file_path="data/olympic_wiki.md"):
    with open(file_path, "r", encoding="utf-8") as fr:
        olympic_text = fr.read()

    # Split
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "H1"),
            ("##", "H2"),
            ("###", "H3"),
        ],
    )

    return splitter.split_text(olympic_text)

In [3]:
#################################################################
# Vector DB 연결
# retriever 생성
#################################################################

def get_vectorstore(collection_name: str = "olympic_info_wiki"):


    dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

    client = QdrantClient(url="http://localhost:6333")

    # 컬렉션 삭제
    if client.collection_exists(collection_name):
        result = client.delete_collection(collection_name=collection_name)

    # 컬렉션 생성
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
      
    )

    vectorstore = QdrantVectorStore(
        client=client,
        collection_name=collection_name,    
        embedding=dense_embeddings
    )
    
    ######################################
    # Document들 추가
    ######################################
    documents = load_and_split_olympic_data()
    vectorstore.add_documents(documents=documents)

    return vectorstore


def get_retriever(vectorstore, k: int = 5):
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )
    return retriever

In [ ]:
vectorstore = get_vectorstore()

retriever = get_retriever(vectorstore)
retriever

In [4]:
################################################################################
# 평가할 RAG Chain
################################################################################

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from operator import itemgetter

vectorstore = get_vectorstore()
retriever = get_retriever(vectorstore)

prompt_txt = """<instruction>
당신은 정보제공을 목적으로하는 유능한 AI Assistant 입니다.
주어진 context의 내용을 기반으로 질문에 답변을 합니다.
Context에 질문에 대한 명확한 정보가 있는 경우 그것을 바탕으로 답변을 합니다.
Context에 질문에 대한 명확한 정보가 없는 경우 "정보가 부족해 답을 할 수없습니다." 라고 답합니다.
절대 추측이나 일반 상식을 바탕으로 답을 하거나 Context 없는 내용을 만들어서 답변해서는 안됩니다.
</instruction>
<context>
{context}
</context>
<question>
{query}
</question>
"""
prompt = ChatPromptTemplate.from_template(
    template=prompt_txt
)

model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()

def format_doc_to_str(documents:list[Document])->list[str]:
    """
    VectorStore에 조회한 문서들(list[Document])에서 내용(page_content)만 추출해서 list[str] 로 반환.
    RAGAS 평가시 context는 각 검색한 문서를 list[str] 로 받기 때문에 이렇게 처리.
    
    Args:
        documents(list[Document]): [Document(..), Document(...), ..]}
    Returns:
        list[str]: 각 문서의 내용만 추출해서 리스트에 담는다.
    """
    return [doc.page_content for doc in documents]

# RAG 체인 -> RAG 평가데이터셋을 만드는 RAG Chain
#          -> 최종응답: LLM의 응답(str), 검색한 문서들(list[str])
chain = RunnablePassthrough() | {
    "context":retriever | format_doc_to_str,
    "query":RunnablePassthrough()
} | { 
    "response": prompt | model | parser,
    "retrieved_context": itemgetter("context")
}

# RunnablePassthrough() -> LCEL 체인을 만들려면 구성요소중 하나가 Runnable이여야 하는데
# 이 체인은 dict | dict 구조개 때문에 앞에 추가.

In [5]:
res = chain.invoke("1회 올림픽은 언제 어디서 열렸지")

In [6]:
print(res.keys())

dict_keys(['response', 'retrieved_context'])


In [7]:
res['response']

'1896년에 그리스 아테네에서 제1회 올림픽이 열렸습니다.'

In [8]:
res['retrieved_context']

['고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이 모여 벌인 일련의 시합이었으며, 육상 경기가 주 종목이지만 격투기와 전차 경기도 열렸다. 그리고 패배하면 죽기도 하였다. 고대 올림픽의 유래는 수수께끼로 남아있다. 잘 알려진 신화로는 헤라클레스와 그의 아버지인 제우스가 올림픽의 창시자였다는 것이다. 전설에 따르면 이 경기를 최초로 \'올림픽\'이라고 부르고, 4년마다 대회를 개최하는 관례를 만든 사람이 헤라클레스라고 한다. 어떤 전설에서는 헤라클레스가 이른바 헤라클레스의 12업을 달성한 뒤에 제우스를 기리고자 올림픽 경기장을 지었다고 한다. 경기장이 완성되자 헤라클레스는 일직선으로 200 걸음을 걸었으며, 이 거리를 "스타디온"이라 불렀는데, 후에 이것이 길이 단위인 \'스타디온\'(그리스어: στάδιον → 라틴어: 영어: stadium)이 되었다. 또 다른 설로는 \'올림픽 휴전\'(그리스어: ἐκεχειρία 에케케이리아[*])이라는 고대 그리스의 관념이 최초의 올림피아 경기와 관련이 있다고 한다. \'올림픽 휴전\'이란 어느 도시 국가라도 올림피아 경기 기간 중에 다른 나라를 침범하면 그에 대한 응징을 받을 수 있다는 뜻으로, "올림픽 기간에는 전쟁하지 말 것"으로 요약할 수 있다.  \n고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에서 우승한 사람이라고 한다.  \n고대 올림피아 경기는 근본적으로 종교적인 중요성을 띄고 있었는데, 스포츠 경기를 할 때는 제우스(올림피아의 제우스 신전에는 페이디아스가 만든 제우스 

# RAGAS 를 이용해 평가를 위한 합성 데이터 셋 만들기

- 평가 데이터셋 구성
  - `user_input`: 사용자 질문
  - `retrieved_contexts`: Vectorstore에서 검색한 context
  - `response`: LLM의 응답
  - `reference`: 정답

## TestsetGenerator
- **문서(retrieved_contexts)를 기준**으로 **질문**, **정답** 을 생성한다.
- 평가할 LLM으로 생성된 질문을 넣어 답변을 추출하여 데이터셋을 구성한다.


> **주의**
> - TestsetGenerator import 시 `No Module named langchain_community.chat_models.vertexai` Error 발생 
> - RAGAS와 langchain-community의 버전 호환성 문제 때문에 발생한다.
> - 해결
>   1. langchain_google_vertexai 설치
>       - `!uv pip install langchain_google_vertexai`
>   2. `.venv\Lib\site-packages\langchain_community\chat_models` 디렉토리 아래 `vertexai.py` 파일을 만들고 아래 코드를 > 넣는다.
>    ```python
>       try:
>           from langchain_google_vertexai import ChatVertexAI
>       except ImportError:
>           class ChatVertexAI:
>               def __init__(self, *args, **kwargs):
>                   raise ImportError(
>                       "ChatVertexAI requires langchain-google-vertexai. "
>                       "Install with: pip install langchain-google-vertexai"
>                   )
>    ```

In [9]:
# 주피터노트북 환경에서 비동기적 처리 위해
# script(.py) 로 작성할 경우는 필요 없다.

import nest_asyncio
nest_asyncio.apply()

In [11]:
##################################
# testset -> Context들(문서들) - [질문 - 정답답변 + Retriever가 찾은 문서 + LLM 응답: chain 생성)]
# 1. Context(문서들)을 추출 - TestsetGenerator -> 질문과 정답답변 생성.

# 데이터셋을 생성할 때 사용할 Context를 추출.
import random
client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "olympic_info_wiki"

# 전체 저장된 문서 중에서 K개만 sampling
info = client.get_collection(COLLECTION_NAME)
total_docs = info.points_count # 총 문서 개수 조회

results, _= client.scroll(
    collection_name=COLLECTION_NAME,
    limit=total_docs
)
# 랜덤하게 K(5)개를 sampling
sample_docs:"list[PointStruct]" = random.sample(results, 5) # 리스트에서 랜덤하게 k(5)개를 추출

# PointStruct - payload: page-content, metadata
# page_content만 추출해서 list[str]
docs = [point.payload['page_content'] for point in sample_docs]

In [12]:
docs

["올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다.  \n#### 개막식  \n개막식 때는 올림픽 헌장에 따라 다양한 행사가 열린다. 개막식의 기본 토대는 벨기에 안트베르펜에서 열린 1920년 하계 올림픽 때 만들어졌다. 개막식은 대개 개최국의 국기가 게양되고 국가가 울려퍼지며 시작된다. 그 후에 개최국이 준비한 그들의 문화를 대표하는 음악, 춤, 영상 따위가 공연된다. 개막식은 아름답기가 매회가 지날수록 웅대해지고 복잡해지는데, 이는 전 대회보다 사람들의 기억에 오래도록 남기기 위함이다. 보도에 의하면 2008 베이징 올림픽 개막식 때 든 비용은 1억 달러로 그 대부분이 예술적인 부분에 들었다고 한다.  \n행사가 끝나면 다음에는 각국의 선수단이 입장한다. 올림픽의 발상지라는 영예를 가진 그리스가 전통적으로 맨 처음에 입장한다. 나머지 각국 선수단은 주최국에서 선택한 언어의 사전 순으로 입장하고 나서, 개최국 선수단이 제일 마지막에 입장한다. 그리스 아테네에서 열린 2004년 하계 올림픽에서는 그리스 국기가 맨 처음에 입장하고, 그리스 선수단이 맨 마지막에 입장했다. 개막식 마지막에는 올림픽 성화가 들어오고 마지막 성화 봉송자에게 전달할 때까지 돌게 된다. 마지막 성화 봉송자는 대체로 유명하고 올림픽에서 성공한 개최국의 선수가 하며 올림픽 성화를 경기장 내에 점화한다. 그 다음 올림픽조직위원장과 IOC 위원장이 개막 선언을 하는데 공식적으로 개막되었다는 것과 올림픽 성화가 점화되는 것을 선언한다. 그 다음에 올림픽조직위원장과 IOC 위원장이 개회사를 낭독하게 된다. 마자막으로 올림픽기 게양에 이어 올림픽 선서를 끝으로 개막식 과정은 모두 끝나게 된다. 2020년 하계 올림픽 이후로는 그리스 - 난민 올림픽 선수단 - 나머지 각국 선수단 - 차차기 올림픽 개최국 - 차기 올림픽 개최국 - 개최국 순서대로 입장한다.  \n#### 폐막식  \n폐막식은 올림픽 경기가 모두 끝난 후에 열린다. 각국의 기수가 경기장에 들어온 후에 국적에 상관

In [ ]:
total_docs
sample_docs

25

In [16]:
# 테스트 셋을 생성
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# TestsetGenerator는 gpt-5 이후 버전은 사용할 수 없다.
## Langchain의 LLM 모델과 Embedding 모델 -> RAGAS에서 사용할 수 있도록 변환(Wrapping)
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    llm_context="""
- 사람들이 올림픽에 대해서 궁금해 할 만한 질문들을 생성한다.
- 데이터셋은 반드시 한국어로 작성한다.
- 데이터셋은 JSON 문법을 지켜서 작성한다. 특히 구두점은 꼭 지켜야 한다.
- 생성된 내용이나 Document에 JSON문법에 맞지 않는 표현이 있으면 반드시 수정해서 처리한다.
""" # 질문/답변을 생성할 때 LLM에게 전달할 System Prompt 를 설정
)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_20456\707878625.py:10: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_20456\707878625.py:11: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


In [17]:
testset = generator.generate_with_chunks(
    docs, testset_size=10, # Context 내용, 테스트셋 개수 (질문-답변 개수)
)

Applying SummaryExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

Node c83326f6-1ff0-41c2-992c-4a73abe78f92 does not have a summary. Skipping filtering.
Node 6b4c8156-2c21-40ad-aeeb-fbef473ff3ae does not have a summary. Skipping filtering.
Node d329b67b-6b09-4998-82fc-d0ead8ca3600 does not have a summary. Skipping filtering.
Node 6e785c63-ef82-4937-8f4c-c06dfaac4e68 does not have a summary. Skipping filtering.
Node ad557e8c-5a92-401c-934e-9cd3c50d2867 does not have a summary. Skipping filtering.


Applying EmbeddingExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [18]:
testset

Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input='올림픽 개막식에서 그리스 선수단이 항상 맨 처음에 입장하는 이유가 뭐에요? 그리고 2020년 하계 올림픽 이후로는 입장 순서가 어떻게 바뀌었나요?', retrieved_contexts=None, reference_contexts=["올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다.  \n#### 개막식  \n개막식 때는 올림픽 헌장에 따라 다양한 행사가 열린다. 개막식의 기본 토대는 벨기에 안트베르펜에서 열린 1920년 하계 올림픽 때 만들어졌다. 개막식은 대개 개최국의 국기가 게양되고 국가가 울려퍼지며 시작된다. 그 후에 개최국이 준비한 그들의 문화를 대표하는 음악, 춤, 영상 따위가 공연된다. 개막식은 아름답기가 매회가 지날수록 웅대해지고 복잡해지는데, 이는 전 대회보다 사람들의 기억에 오래도록 남기기 위함이다. 보도에 의하면 2008 베이징 올림픽 개막식 때 든 비용은 1억 달러로 그 대부분이 예술적인 부분에 들었다고 한다.  \n행사가 끝나면 다음에는 각국의 선수단이 입장한다. 올림픽의 발상지라는 영예를 가진 그리스가 전통적으로 맨 처음에 입장한다. 나머지 각국 선수단은 주최국에서 선택한 언어의 사전 순으로 입장하고 나서, 개최국 선수단이 제일 마지막에 입장한다. 그리스 아테네에서 열린 2004년 하계 올림픽에서는 그리스 국기가 맨 처음에 입장하고, 그리스 선수단이 맨 마지막에 입장했다. 개막식 마지막에는 올림픽 성화가 들어오고 마지막 성화 봉송자에게 전달할 때까지 돌게 된다. 마지막 성화 봉송자는 대체로 유명하고 올림픽에서 성공한 개최국의 선수가 하며 올림픽 성화를 경기장 내에 점화한다. 그 다음 올림픽조직위원장과 IOC 위원장이 개막 선언을 하는데 공식적으로 개막되었다는 것과 올림픽 성화가 점화되는 것을 선언한다. 그 다음에 올림픽조직위원장과 IOC 위원장이 개회사를 낭독하게 된다. 마자막으로 올림픽

In [29]:
sample1 = testset.samples[2].eval_sample # 10개중 첫번째 테스트데이터
print("사용자질문:", sample1.user_input)
print("Context:", sample1.reference_contexts)   # 질문과 답변을 만들 때 사용한 context (검색시 찾아야 하는 문서)
print("생성된답변(정답):", sample1.reference)
########### 평가 대상 RAG system을 이용해서 채워 넣어야 한다.
print("평가대상 RAG의 답변:", sample1.response)
print("평가대상 RAG가 검색한 Context:", sample1.retrieved_contexts)

사용자질문: 헬싱키 올림픽 언제였나요?
Context: ["쿠베르탱이 말했던 원래 이념과는 반대로 올림픽이 정치 혹은 체제 선전의 장으로 이용되는 경우가 있었다. 1936년 하계 올림픽을 개최할 때 당시의 나치독일은 나치는 자비롭고 평화를 위한다는 것을 설명하고 싶어했다. 또 이 올림픽에서 아리안족의 우월함을 보여줄 생각이었으나 이는 흑인이었던 제시 오언스가 금메달을 4개나 따내면서 실현되지는 못했다. 소련은 헬싱키에서 열린 1952년 하계 올림픽 때 처음으로 참가했다. 그 전에는 소련이 조직한 스파르타키아다라는 대회에 1928년부터 참가했었다. 다른 공산주의 국가들은 1920년대와 1930년대의 전쟁 기간 사이에 노동자 올림픽(Socialist Workers' Sport International)을 조직했는데, 이는 올림픽을 자본가와 귀족들의 대회로 여기고 그에 대한 대안으로 고안된 대회였다. 그 이후 소련은 1956년 하계 올림픽부터 1988년 하계 올림픽까지 엄청난 스포츠강국의 면모를 보여주며 올림픽에서의 명성을 드높였다.  \n선수 개인이 자신의 정치적 성향에 대해 표현하기도 했다. 멕시코 시티에서 열린 1968년 하계 올림픽의 육상부문 200m 경기에서 각각 1위와 3위를 한 미국의 토미 스미스와 존 카를로스는 시상식 때 블랙 파워 설루트(Black Power salute , 흑인 차별 반대 행위)를 선보였으며 2위를 한 피터 노먼도 상황을 깨닫고 스미스와 카를로스의 행위를 지지한다는 뜻에서 급하게 인권을 위한 올림픽 프로젝트(OPHR) 배지를 달았다. 이 사건에 대해서 IOC 위원장이었던 에이버리 브런디지는 미국 올림픽 위원회에 이 두 선수를 미국으로 돌려보내거나 미국 육상팀 전부를 돌려보내는 둘 중 하나의 선택을 하게 했고, 미국 올림픽 위원회는 두 선수를 미국으로 돌려 보낸다.  \n현재 이란 정부는 이스라엘과의 어떤 경기 경쟁이든 피하고 있다. 2008년 하계 올림픽 때 이란의 수영 선수는 이스라엘 수영 선수와 같이 경기한다는 이유로 경기를 포

In [30]:
# 생성된 Testset을 Pandas DataFrame으로 변환

eval_df = testset.to_pandas()
eval_df.shape

(10, 7)

In [31]:
eval_df.head(3)

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,올림픽 개막식에서 그리스 선수단이 항상 맨 처음에 입장하는 이유가 뭐에요? 그리고 ...,"[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...",올림픽 개막식에서 그리스는 올림픽의 발상지라는 영예를 가지고 있기 때문에 전통적으로...,Sports Marketing Historian,MISSPELLED,LONG,single_hop_specific_query_synthesizer
1,미디어 올림픽에 뭐 했어?,[처음에 IOC는 스폰서에게서 자금제공을 받는 것을 거부했었다. 이런 방침은 에이버...,IOC는 텔레비전 같은 미디어들이 갖는 잠재성과 큰 수익을 가져오는 광고시장에 대해...,Olympic Enthusiast,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
2,헬싱키 올림픽 언제였나요?,[쿠베르탱이 말했던 원래 이념과는 반대로 올림픽이 정치 혹은 체제 선전의 장으로 이...,소련은 헬싱키에서 열린 1952년 하계 올림픽 때 처음으로 참가했다.,Olympic History Researcher,MISSPELLED,SHORT,single_hop_specific_query_synthesizer


In [ ]:
row_idx = 0
q = eval_df.loc[row_idx, 'user_input']  # 질문 조회
resp = chain.invoke(q)  # dict[response, retrieved_context]

In [35]:
resp['response']

'그리스 선수단이 개막식에서 항상 맨 처음 입장하는 이유는 **그리스가 올림픽의 발상지라는 영예를 가지고 있기 때문**입니다.\n\n그리고 **2020년 하계 올림픽 이후**에는 입장 순서가 다음과 같이 바뀌었습니다.\n\n**그리스 → 난민 올림픽 선수단 → 나머지 각국 선수단 → 차차기 올림픽 개최국 → 차기 올림픽 개최국 → 개최국** 순서입니다.'

In [36]:
resp['retrieved_context']

["올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다.  \n#### 개막식  \n개막식 때는 올림픽 헌장에 따라 다양한 행사가 열린다. 개막식의 기본 토대는 벨기에 안트베르펜에서 열린 1920년 하계 올림픽 때 만들어졌다. 개막식은 대개 개최국의 국기가 게양되고 국가가 울려퍼지며 시작된다. 그 후에 개최국이 준비한 그들의 문화를 대표하는 음악, 춤, 영상 따위가 공연된다. 개막식은 아름답기가 매회가 지날수록 웅대해지고 복잡해지는데, 이는 전 대회보다 사람들의 기억에 오래도록 남기기 위함이다. 보도에 의하면 2008 베이징 올림픽 개막식 때 든 비용은 1억 달러로 그 대부분이 예술적인 부분에 들었다고 한다.  \n행사가 끝나면 다음에는 각국의 선수단이 입장한다. 올림픽의 발상지라는 영예를 가진 그리스가 전통적으로 맨 처음에 입장한다. 나머지 각국 선수단은 주최국에서 선택한 언어의 사전 순으로 입장하고 나서, 개최국 선수단이 제일 마지막에 입장한다. 그리스 아테네에서 열린 2004년 하계 올림픽에서는 그리스 국기가 맨 처음에 입장하고, 그리스 선수단이 맨 마지막에 입장했다. 개막식 마지막에는 올림픽 성화가 들어오고 마지막 성화 봉송자에게 전달할 때까지 돌게 된다. 마지막 성화 봉송자는 대체로 유명하고 올림픽에서 성공한 개최국의 선수가 하며 올림픽 성화를 경기장 내에 점화한다. 그 다음 올림픽조직위원장과 IOC 위원장이 개막 선언을 하는데 공식적으로 개막되었다는 것과 올림픽 성화가 점화되는 것을 선언한다. 그 다음에 올림픽조직위원장과 IOC 위원장이 개회사를 낭독하게 된다. 마자막으로 올림픽기 게양에 이어 올림픽 선서를 끝으로 개막식 과정은 모두 끝나게 된다. 2020년 하계 올림픽 이후로는 그리스 - 난민 올림픽 선수단 - 나머지 각국 선수단 - 차차기 올림픽 개최국 - 차기 올림픽 개최국 - 개최국 순서대로 입장한다.  \n#### 폐막식  \n폐막식은 올림픽 경기가 모두 끝난 후에 열린다. 각국의 기수가 경기장에 들어온 후에 국적에 상관

In [37]:
# testset의 모든 데이터에 llm 응답과 retriever의 검색결과 추가
response_list = [] # LLM 응답들을 저장할 리스트
retrieved_context_list = [] # retriever가 검색한 문서들을 저장할 리스트

for user_input in eval_df['user_input']:
    resp = chain.invoke(user_input)
    response_list.append(resp['response'])
    retrieved_context_list.append(resp['retrieved_context'])

In [38]:
print(len(response_list), len(retrieved_context_list))

10 10


In [39]:
response_list

['그리스 선수단이 항상 맨 처음에 입장하는 이유는, **그리스가 올림픽의 발상지라는 영예를 가지고 있기 때문**입니다.\n\n또한 **2020년 하계 올림픽 이후**부터는 입장 순서가 다음과 같이 바뀌었습니다:  \n**그리스 - 난민 올림픽 선수단 - 나머지 각국 선수단 - 차차기 올림픽 개최국 - 차기 올림픽 개최국 - 개최국** 순서로 입장합니다.',
 '정보가 부족해 답을 할 수없습니다.',
 '정보가 부족해 답을 할 수없습니다.',
 '정보가 부족해 답을 할 수없습니다.',
 '정보가 부족해 답을 할 수없습니다.',
 '20세기 초반에는 많은 운동 선수들이 **기록 향상**을 위해 약물을 복용하기 시작했습니다. 예로 1904년 하계 올림픽 마라톤 우승자 토머스 J. 힉스는 코치에게서 **스트리크닌과 브랜디**를 받았습니다.\n\n이후 올림픽에서는 약물의 위험성과 부작용이 점차 문제로 인식되었고, **1960년 로마 대회에서 암페타민 과다 복용으로 사망한 사례**까지 나오면서 각 경기 연맹이 **1960년대 중반부터 약물 복용 금지**를 시작했습니다. 이어서 **1967년에는 IOC도 약물 복용 금지에 동참**했습니다.\n\n즉, 토머스 J. 힉스의 사례 같은 초기 약물 사용은 선수들 사이에서 기록 향상을 위한 약물 복용이 널리 퍼져 있음을 보여주었고, 이후 사망 사례와 함께 도핑 금지 정책이 만들어지고 강화되는 흐름으로 이어졌습니다.',
 '1936년 하계 올림픽은 나치독일이 **나치는 자비롭고 평화를 위한다는 이미지**를 세계에 보여주고, 동시에 **아리안족의 우월함**을 입증하는 데 정치적으로 이용했습니다.  \n\n그러나 그 결과는 뜻대로 되지 않았습니다. 흑인 선수 **제시 오언스가 금메달 4개**를 따내면서 아리안족 우월성을 드러내려던 목적은 실현되지 않았습니다.',
 '1960년 하계 올림픽은 로마에서 열렸고, 구트만이 400명의 선수들을 “Parallel Olympics”에 참가시킨 대회가 있었으며 이것이 곧 1회 패럴림픽으로 알려지게 

In [40]:
retrieved_context_list

[["올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다.  \n#### 개막식  \n개막식 때는 올림픽 헌장에 따라 다양한 행사가 열린다. 개막식의 기본 토대는 벨기에 안트베르펜에서 열린 1920년 하계 올림픽 때 만들어졌다. 개막식은 대개 개최국의 국기가 게양되고 국가가 울려퍼지며 시작된다. 그 후에 개최국이 준비한 그들의 문화를 대표하는 음악, 춤, 영상 따위가 공연된다. 개막식은 아름답기가 매회가 지날수록 웅대해지고 복잡해지는데, 이는 전 대회보다 사람들의 기억에 오래도록 남기기 위함이다. 보도에 의하면 2008 베이징 올림픽 개막식 때 든 비용은 1억 달러로 그 대부분이 예술적인 부분에 들었다고 한다.  \n행사가 끝나면 다음에는 각국의 선수단이 입장한다. 올림픽의 발상지라는 영예를 가진 그리스가 전통적으로 맨 처음에 입장한다. 나머지 각국 선수단은 주최국에서 선택한 언어의 사전 순으로 입장하고 나서, 개최국 선수단이 제일 마지막에 입장한다. 그리스 아테네에서 열린 2004년 하계 올림픽에서는 그리스 국기가 맨 처음에 입장하고, 그리스 선수단이 맨 마지막에 입장했다. 개막식 마지막에는 올림픽 성화가 들어오고 마지막 성화 봉송자에게 전달할 때까지 돌게 된다. 마지막 성화 봉송자는 대체로 유명하고 올림픽에서 성공한 개최국의 선수가 하며 올림픽 성화를 경기장 내에 점화한다. 그 다음 올림픽조직위원장과 IOC 위원장이 개막 선언을 하는데 공식적으로 개막되었다는 것과 올림픽 성화가 점화되는 것을 선언한다. 그 다음에 올림픽조직위원장과 IOC 위원장이 개회사를 낭독하게 된다. 마자막으로 올림픽기 게양에 이어 올림픽 선서를 끝으로 개막식 과정은 모두 끝나게 된다. 2020년 하계 올림픽 이후로는 그리스 - 난민 올림픽 선수단 - 나머지 각국 선수단 - 차차기 올림픽 개최국 - 차기 올림픽 개최국 - 개최국 순서대로 입장한다.  \n#### 폐막식  \n폐막식은 올림픽 경기가 모두 끝난 후에 열린다. 각국의 기수가 경기장에 들어온 후에 국적에 상

In [44]:
####################################
# eval_df 에 컬럼으로 추가
####################################
eval_df['response'] = response_list
eval_df['retrieved_contexts'] = retrieved_context_list
eval_df.head()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name,response,retrieved_contexts
0,올림픽 개막식에서 그리스 선수단이 항상 맨 처음에 입장하는 이유가 뭐에요? 그리고 ...,"[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...",올림픽 개막식에서 그리스는 올림픽의 발상지라는 영예를 가지고 있기 때문에 전통적으로...,Sports Marketing Historian,MISSPELLED,LONG,single_hop_specific_query_synthesizer,"그리스 선수단이 항상 맨 처음에 입장하는 이유는, **그리스가 올림픽의 발상지라는 ...","[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#..."
1,미디어 올림픽에 뭐 했어?,"[올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)...",IOC는 텔레비전 같은 미디어들이 갖는 잠재성과 큰 수익을 가져오는 광고시장에 대해...,Olympic Enthusiast,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,"[올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)..."
2,헬싱키 올림픽 언제였나요?,[올림픽 개최지는 해당 올림픽 개최 7년 전에 IOC 위원들의 투표로 결정된다. 개...,소련은 헬싱키에서 열린 1952년 하계 올림픽 때 처음으로 참가했다.,Olympic History Researcher,MISSPELLED,SHORT,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,[올림픽 개최지는 해당 올림픽 개최 7년 전에 IOC 위원들의 투표로 결정된다. 개...
3,한스 군나르 리렌바르 뭐 했어?,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...",한스 군나르 리렌바르는 1968년 하계 올림픽 근대 5종 경기에 출전해 동메달을 땄...,Sports Marketing Historian,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실..."
4,런던 올림픽이 패럴림픽과 어떤 관련이 있나요?,[패럴림픽(Paralympic)은 신체·감각 장애가 있는운동 선수가 참가하는 국제 ...,1948년 런던 올림픽과 동시에 루드비히 구트만 경은 몇몇 병원들을 연합해서 여러 ...,Sports Marketing Historian,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,[패럴림픽(Paralympic)은 신체·감각 장애가 있는운동 선수가 참가하는 국제 ...


In [45]:
# eval_df 를 RAGAS의 평가데이터셋 타입으로 변환.
from ragas import EvaluationDataset
eval_dataset = EvaluationDataset.from_pandas(
    eval_df[['user_input', 'retrieved_contexts', 'response', 'reference']]
)
eval_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=10)

In [50]:
################################
# 평가
################################
from ragas.metrics import(
    LLMContextRecall, # Context Recall
    LLMContextPrecisionWithReference, # Context Precision
    Faithfulness,
    AnswerRelevancy
)
from ragas import evaluate

C:\Users\Playdata\AppData\Local\Temp\ipykernel_20456\1539178808.py:4: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import(
C:\Users\Playdata\AppData\Local\Temp\ipykernel_20456\1539178808.py:4: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import(
C:\Users\Playdata\AppData\Local\Temp\ipykernel_20456\1539178808.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import(
C:\Users\P

In [52]:
# 평가할 때 사용할 LLM, Embedding 모델
eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

# Metric(평가지표) 객체를 List로 묶어준다.
## 내가 평가할 지표들만 묶어준다.
metrics = [
    LLMContextRecall(llm=eval_llm),
    LLMContextPrecisionWithReference(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings=eval_embeddings)
]
# 평가진행
eval_result = evaluate(dataset=eval_dataset, metrics=metrics)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_20456\2225221690.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_20456\2225221690.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Exception raised in Job[2]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[16]: TimeoutError()
Exception raised in Job[18]: TimeoutError()
Exception raised in Job[20]: TimeoutError()
Exception raised in Job[22]: TimeoutError()
Exception raised in Job[24]: TimeoutError()
Exception raised in Job[26]: TimeoutError()
Exception raised in Job[30]: TimeoutError()


In [53]:
eval_result

{'context_recall': 0.5714, 'llm_context_precision_with_reference': 0.7333, 'faithfulness': 0.6667, 'answer_relevancy': 0.4484}

In [ ]:
# print(type(eval_result))
## 개별 평가데이터에 대한 평가점수.
result_df = eval_result.to_pandas()

<class 'ragas.dataset_schema.EvaluationResult'>


In [55]:
result_df

,user_input,retrieved_contexts,response,reference,context_recall,llm_context_precision_with_reference,faithfulness,answer_relevancy
0,올림픽 개막식에서 그리스 선수단이 항상 맨 처음에 입장하는 이유가 뭐에요? 그리고 ...,"[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...","그리스 선수단이 항상 맨 처음에 입장하는 이유는, **그리스가 올림픽의 발상지라는 ...",올림픽 개막식에서 그리스는 올림픽의 발상지라는 영예를 가지고 있기 때문에 전통적으로...,1.0,1.000000,NaN,0.839811
1,미디어 올림픽에 뭐 했어?,"[올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)...",정보가 부족해 답을 할 수없습니다.,IOC는 텔레비전 같은 미디어들이 갖는 잠재성과 큰 수익을 가져오는 광고시장에 대해...,0.0,0.333333,NaN,0.000000
2,헬싱키 올림픽 언제였나요?,[올림픽 개최지는 해당 올림픽 개최 7년 전에 IOC 위원들의 투표로 결정된다. 개...,정보가 부족해 답을 할 수없습니다.,소련은 헬싱키에서 열린 1952년 하계 올림픽 때 처음으로 참가했다.,0.0,0.000000,0.0,0.000000
3,한스 군나르 리렌바르 뭐 했어?,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...",정보가 부족해 답을 할 수없습니다.,한스 군나르 리렌바르는 1968년 하계 올림픽 근대 5종 경기에 출전해 동메달을 땄...,0.0,0.000000,NaN,0.000000
4,런던 올림픽이 패럴림픽과 어떤 관련이 있나요?,[패럴림픽(Paralympic)은 신체·감각 장애가 있는운동 선수가 참가하는 국제 ...,정보가 부족해 답을 할 수없습니다.,1948년 런던 올림픽과 동시에 루드비히 구트만 경은 몇몇 병원들을 연합해서 여러 ...,NaN,1.000000,NaN,0.000000
5,20세기 초반 올림픽에서 약물 복용이 시작된 배경과 1904년 하계 올림픽에서 토머...,"[20세기 초반, 많은 운동 선수들은 기록향상을 위해 약물을 복용하기 시작했다. 예...",20세기 초반에는 많은 운동 선수들이 **기록 향상**을 위해 약물을 복용하기 시작...,"20세기 초반, 많은 운동 선수들이 기록 향상을 위해 약물을 복용하기 시작했으며, ...",NaN,1.000000,NaN,0.589924
6,"1936년 하계 올림픽이 정치적으로 어떻게 이용됐고, 그 결과는 어땠나요?",[쿠베르탱이 말했던 원래 이념과는 반대로 올림픽이 정치 혹은 체제 선전의 장으로 이...,1936년 하계 올림픽은 나치독일이 **나치는 자비롭고 평화를 위한다는 이미지**를...,1936년 하계 올림픽은 나치독일이 나치는 자비롭고 평화를 위한다는 것을 설명하고 ...,NaN,1.000000,NaN,0.865720
7,1960년 하계 올림픽이 패럴림픽 역사에서 왜 중요해?,[패럴림픽(Paralympic)은 신체·감각 장애가 있는운동 선수가 참가하는 국제 ...,"1960년 하계 올림픽은 로마에서 열렸고, 구트만이 400명의 선수들을 “Paral...","1960년 하계 올림픽 때 구트만이 400명의 선수를 ""Parallel Olympi...",1.0,1.000000,NaN,0.778316
8,1988년 하계 올림픽이 패럴림픽 개최 방식에 어떤 변화를 가져왔는가?,[패럴림픽(Paralympic)은 신체·감각 장애가 있는운동 선수가 참가하는 국제 ...,1988년 서울 하계 올림픽부터는 **하계 올림픽을 개최한 도시가 패럴림픽도 함께 ...,1988년 하계 올림픽부터는 하계 올림픽을 개최한 도시가 패럴림픽도 같이 개최하기로...,1.0,1.000000,1.0,0.734098
9,"1952년 하계 올림픽에서 소련이 처음 참가한 거, 그리고 그 전에는 소련이 왜 스...",[쿠베르탱이 말했던 원래 이념과는 반대로 올림픽이 정치 혹은 체제 선전의 장으로 이...,1952년 하계 올림픽에서 소련은 처음으로 참가했습니다. 그 전에는 소련이 1928...,소련은 1952년 하계 올림픽에서 처음으로 참가했어. 그 전에는 1928년부터 소련...,1.0,1.000000,1.0,0.676171
